# 08 — Acne / Lesion Detection (YOLOv8s)

**Architecture:** YOLOv8s (COCO pretrained) fine-tuned for 6-class lesion detection

**Classes:** comedone, papule, pustule, nodule, macule, patch

**Training strategy:**
- **Phase 1 — General training:** 50 epochs, lr0=1e-3 (full model)
- **Phase 2 — Fine-tuning:** 30 epochs, lr0=5e-4, freeze first 10 layers

**Export:** ONNX + CoreML

In [ ]:
# ============================================================
# Cell 1: Install dependencies
# ============================================================
!pip install -q ultralytics onnx onnxruntime coremltools

In [ ]:
# ============================================================
# Cell 2: Imports & configuration
# ============================================================
import os
from pathlib import Path
from ultralytics import YOLO
import torch
import yaml

# ---------- paths ----------
DATA_ROOT    = Path('/root/cornell-hackathon/ml/data/lesion')
DATASET_YAML = DATA_ROOT / 'dataset.yaml'
OUTPUT_DIR   = Path('/root/cornell-hackathon/ml/outputs/lesion')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- verify dataset ----------
assert DATASET_YAML.exists(), f'dataset.yaml not found at {DATASET_YAML}'
with open(DATASET_YAML) as f:
    ds_cfg = yaml.safe_load(f)

print(f'Dataset YAML: {DATASET_YAML}')
print(f'  path:  {ds_cfg["path"]}')
print(f'  train: {ds_cfg["train"]}')
print(f'  val:   {ds_cfg["val"]}')
print(f'  test:  {ds_cfg.get("test", "N/A")}')
print(f'  nc:    {ds_cfg["nc"]}')
print(f'  names: {ds_cfg["names"]}')

# Count images
for split in ['train', 'val', 'test']:
    img_dir = DATA_ROOT / split / 'images'
    if img_dir.exists():
        n = len(list(img_dir.glob('*')))
        print(f'  {split} images: {n}')

print(f'\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# ============================================================
# Cell 3: Phase 1 — General training (50 epochs)
# ============================================================

# Load COCO-pretrained YOLOv8s
model_p1 = YOLO('yolov8s.pt')

# Phase 1: Train all layers
results_p1 = model_p1.train(
    data=str(DATASET_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
    lr0=1e-3,
    lrf=0.01,
    optimizer='AdamW',
    weight_decay=5e-4,
    warmup_epochs=3,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    patience=15,
    project=str(OUTPUT_DIR),
    name='phase1_general',
    exist_ok=True,
    verbose=True,
    seed=42,
)

print('\n=== Phase 1 complete ===')

In [ ]:
# ============================================================
# Cell 4: Phase 1 — Validation metrics
# ============================================================

# Validate Phase 1 best model
p1_best = OUTPUT_DIR / 'phase1_general' / 'weights' / 'best.pt'
print(f'Phase 1 best weights: {p1_best}')

model_p1_best = YOLO(str(p1_best))
metrics_p1 = model_p1_best.val(data=str(DATASET_YAML), split='val')

print(f'\nPhase 1 Validation Results:')
print(f'  mAP50:    {metrics_p1.box.map50:.4f}')
print(f'  mAP50-95: {metrics_p1.box.map:.4f}')
print(f'  Precision: {metrics_p1.box.mp:.4f}')
print(f'  Recall:    {metrics_p1.box.mr:.4f}')

In [ ]:
# ============================================================
# Cell 5: Phase 2 — Fine-tuning (30 epochs, freeze 10 layers)
# ============================================================

# Load the Phase 1 best model
model_p2 = YOLO(str(p1_best))

# Phase 2: Fine-tune with first 10 backbone layers frozen
results_p2 = model_p2.train(
    data=str(DATASET_YAML),
    epochs=30,
    imgsz=640,
    batch=16,
    lr0=5e-4,
    lrf=0.01,
    optimizer='AdamW',
    weight_decay=5e-4,
    warmup_epochs=2,
    freeze=10,
    mosaic=0.5,
    mixup=0.05,
    hsv_h=0.01,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=5.0,
    translate=0.05,
    scale=0.3,
    fliplr=0.5,
    patience=10,
    project=str(OUTPUT_DIR),
    name='phase2_finetune',
    exist_ok=True,
    verbose=True,
    seed=42,
)

print('\n=== Phase 2 complete ===')

In [ ]:
# ============================================================
# Cell 6: Phase 2 — Validation & test metrics
# ============================================================

p2_best = OUTPUT_DIR / 'phase2_finetune' / 'weights' / 'best.pt'
print(f'Phase 2 best weights: {p2_best}')

model_final = YOLO(str(p2_best))

# Validation set
metrics_val = model_final.val(data=str(DATASET_YAML), split='val')
print(f'\nFinal Validation Results:')
print(f'  mAP50:     {metrics_val.box.map50:.4f}')
print(f'  mAP50-95:  {metrics_val.box.map:.4f}')
print(f'  Precision: {metrics_val.box.mp:.4f}')
print(f'  Recall:    {metrics_val.box.mr:.4f}')

# Test set
metrics_test = model_final.val(data=str(DATASET_YAML), split='test')
print(f'\nFinal Test Results:')
print(f'  mAP50:     {metrics_test.box.map50:.4f}')
print(f'  mAP50-95:  {metrics_test.box.map:.4f}')
print(f'  Precision: {metrics_test.box.mp:.4f}')
print(f'  Recall:    {metrics_test.box.mr:.4f}')

# Per-class mAP50
class_names = ds_cfg['names']
print(f'\nPer-class mAP50 (test):')
for i, name in enumerate(class_names):
    if i < len(metrics_test.box.ap50):
        print(f'  {name:12s}: {metrics_test.box.ap50[i]:.4f}')

In [ ]:
# ============================================================
# Cell 7: Visual predictions on test images
# ============================================================
import matplotlib.pyplot as plt
from PIL import Image
import glob
import random

test_images = sorted(glob.glob(str(DATA_ROOT / 'test' / 'images' / '*')))
sample_images = random.sample(test_images, min(6, len(test_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, img_path in enumerate(sample_images):
    results = model_final.predict(img_path, conf=0.25, verbose=False)
    annotated = results[0].plot()
    axes[i].imshow(annotated[:, :, ::-1])  # BGR -> RGB
    axes[i].set_title(Path(img_path).name, fontsize=9)
    axes[i].axis('off')

# Hide unused axes
for j in range(len(sample_images), len(axes)):
    axes[j].axis('off')

plt.suptitle('Lesion Detection — Sample Test Predictions', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'test_predictions.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# Cell 8: Export — ONNX
# ============================================================

onnx_path = model_final.export(
    format='onnx',
    imgsz=640,
    half=False,
    simplify=True,
    opset=14,
    dynamic=True,
)
print(f'ONNX model exported: {onnx_path}')

# Copy to output dir
import shutil
onnx_dest = OUTPUT_DIR / 'lesion_detector.onnx'
shutil.copy2(onnx_path, onnx_dest)
print(f'Copied to: {onnx_dest}')
print(f'File size: {Path(onnx_dest).stat().st_size / 1e6:.1f} MB')

# Quick ONNX validation
import onnx
onnx_model = onnx.load(str(onnx_dest))
onnx.checker.check_model(onnx_model)
print('ONNX validation passed.')

In [ ]:
# ============================================================
# Cell 9: Export — CoreML
# ============================================================

coreml_path = model_final.export(
    format='coreml',
    imgsz=640,
    half=False,
    nms=True,
)
print(f'CoreML model exported: {coreml_path}')

# Copy to output dir
coreml_dest = OUTPUT_DIR / 'lesion_detector.mlpackage'
if Path(coreml_path).is_dir():
    if coreml_dest.exists():
        shutil.rmtree(coreml_dest)
    shutil.copytree(coreml_path, coreml_dest)
else:
    shutil.copy2(coreml_path, coreml_dest)
print(f'Copied to: {coreml_dest}')

In [ ]:
# ============================================================
# Cell 10: Copy final best weights to output dir
# ============================================================

final_pt_dest = OUTPUT_DIR / 'lesion_detector_best.pt'
shutil.copy2(str(p2_best), str(final_pt_dest))
print(f'Final weights saved: {final_pt_dest}')
print(f'File size: {final_pt_dest.stat().st_size / 1e6:.1f} MB')

In [ ]:
# ============================================================
# Cell 11: Summary
# ============================================================
print('=' * 55)
print('LESION DETECTION MODEL TRAINING COMPLETE')
print('=' * 55)
print(f'\nPhase 1 (general):   50 epochs, lr0=1e-3')
print(f'  Val mAP50: {metrics_p1.box.map50:.4f}')
print(f'\nPhase 2 (fine-tune): 30 epochs, lr0=5e-4, freeze=10')
print(f'  Val mAP50:  {metrics_val.box.map50:.4f}')
print(f'  Test mAP50: {metrics_test.box.map50:.4f}')
print(f'\nClasses: {class_names}')
print(f'\nSaved artifacts in {OUTPUT_DIR}:')
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file():
        print(f'  {p.name:40s} ({p.stat().st_size / 1e6:.1f} MB)')
    else:
        print(f'  {p.name + "/":40s} (directory)')